# VisionLeader UFV — Colab Backend

**TFG: Alberto Gómez Soteres — Universidad Francisco de Vitoria**

Este notebook expone dos endpoints para la app de navegación:
- `POST /analyze` — PaliGemma analiza imagen de cartel → edificio + planta
- `POST /transcribe` — Whisper transcribe audio → texto

---

### Antes de ejecutar: añade tus tokens en Colab Secrets

1. Haz click en el icono **🔑 (llave)** en el panel izquierdo de Colab
2. Añade estos dos secrets:

| Nombre | Valor | Dónde conseguirlo |
|--------|-------|-------------------|
| `HF_TOKEN` | `hf_...` | huggingface.co/settings/tokens |
| `NGROK_TOKEN` | `2...` | dashboard.ngrok.com/get-started/your-authtoken |

3. Activa el toggle **"Notebook access"** para cada secret
4. Ejecuta las celdas en orden (**Runtime → Run all**)

> **Nota:** Asegúrate de tener **GPU activada**: Runtime → Change runtime type → T4 GPU

## Paso 1 — Instalar dependencias

In [ ]:
!pip install -q fastapi uvicorn python-multipart
!pip install -q "transformers>=4.40.0" accelerate
!pip install -q Pillow pyngrok openai-whisper
print('All packages installed!')

## Paso 2 — Cargar tokens desde Colab Secrets

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

assert HF_TOKEN, 'HF_TOKEN not found. Add it in the Secrets panel (key icon on the left).'
assert NGROK_TOKEN, 'NGROK_TOKEN not found. Add it in the Secrets panel (key icon on the left).'

print('Secrets loaded correctly.')

## Paso 3 — Cargar modelos (Whisper + PaliGemma)

In [ ]:
import torch
import whisper

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print('\nLoading Whisper (base)...')
whisper_model = whisper.load_model('base')
print('Whisper loaded.')

In [ ]:
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from huggingface_hub import login

login(token=HF_TOKEN)

MODEL_ID = 'google/paligemma2-3b-mix-448'
print(f'Loading {MODEL_ID}... (first time takes ~3 min)')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else DEVICE,
    low_cpu_mem_usage=True,
)
model.eval()

print('PaliGemma loaded.')
if DEVICE == 'cuda':
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU memory: {used:.1f} / {total:.1f} GB used')

## Paso 4 — Crear la API

In [ ]:
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import io, os, tempfile

app = FastAPI(title='VisionLeader UFV - Colab Backend')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/')
async def root():
    return {'status': 'running', 'model': MODEL_ID, 'device': DEVICE}


@app.post('/analyze')
async def analyze_image(file: UploadFile = File(...), prompt: str = Form(None), max_tokens: str = Form(None)):
    """Analyze a building sign image with PaliGemma."""
    try:
        image = Image.open(io.BytesIO(await file.read())).convert('RGB')
        max_new = int(max_tokens) if (max_tokens and max_tokens.isdigit()) else 256

        # Default prompt: format FIRST so the model prioritizes it
        prompt_to_use = prompt or (
            'Answer in this exact format: Building: [CODE], Floor: [NUMBER]\n'
            'From this campus sign, find the building code (EPS, CD, or H) after EDIFICIO/BUILDING '
            'and the floor number (-1, 0, 1, or 2) after PLANTA/FLOOR in the top right corner.'
        )
        # PaliGemma espera <image> al inicio (igual que en paligemma_test_stairs.ipynb)
        if not prompt_to_use.strip().startswith('<image>'):
            prompt_to_use = '<image>\n' + prompt_to_use

        print(f'[/analyze] Prompt: {prompt_to_use!r}')

        inputs = processor(
            text=prompt_to_use,
            images=image,
            padding='longest',
            return_tensors='pt',
        ).to(model.device)
        inputs = inputs.to(dtype=model.dtype)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                do_sample=False,
                num_beams=1,
                pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        input_len = inputs['input_ids'].shape[-1]
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        print(f'[/analyze] Response: {response!r}')
        return {'success': True, 'response': response, 'error': None}

    except Exception as e:
        print(f'[/analyze] Error: {e}')
        return {'success': False, 'response': '', 'error': str(e)}


@app.post('/transcribe')
async def transcribe_audio(file: UploadFile = File(...)):
    """Transcribe audio to text with Whisper."""
    try:
        audio_data = await file.read()
        with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as tmp:
            tmp.write(audio_data)
            tmp_path = tmp.name
        try:
            result = whisper_model.transcribe(
                tmp_path,
                language='en',
                fp16=False,
                no_speech_threshold=0.3,
                condition_on_previous_text=False,
            )
            text = result['text'].strip()
            print(f'[/transcribe] Text: {text!r}')
            return {'success': True, 'text': text, 'error': None}
        finally:
            os.remove(tmp_path)

    except Exception as e:
        print(f'[/transcribe] Error: {e}')
        return {'success': False, 'text': '', 'error': str(e)}


print('FastAPI app ready.')

## Paso 5 — Arrancar el servidor y obtener URL pública

In [ ]:
from pyngrok import ngrok, conf
import uvicorn
import threading
import time

PORT = 8001

# Configure ngrok with authtoken
conf.get_default().auth_token = NGROK_TOKEN

# Start FastAPI server in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='error')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)  # Wait for server to start

# Open ngrok tunnel
tunnel = ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url

print('=' * 55)
print('BACKEND READY')
print('=' * 55)
print(f'\nPublic URL: {PUBLIC_URL}')
print(f'\nCopy this URL into planning_api.py (line 95):')
print(f'  COLAB_API_URL = "{PUBLIC_URL}"')
print('\n' + '=' * 55)
print('Keep this notebook running while using the app.')

## Paso 6 — Verificar que funciona (opcional)

In [ ]:
import requests

r = requests.get(f'{PUBLIC_URL}/')
print('Status check:', r.json())